# Example — the graph of a lab test, two ways

How a consumer can use the cdisc-for-all graphs: a SPARQL CONSTRUCT query returns a subgraph, every triple as it is in the graphs. Nothing is added and nothing is written back to the repository.

1. `findings_test_graph.rq` — everything linked to one SDTM CT test code (example: GLUC in Laboratory Test Code): the COSMoS Dataset Specializations that assign it, their variables, codelists and terms, the BCs and DECs, the NCIt concepts with ancestors and CUIs, and (local only) the SDTMIG and SDTM Model variables.
2. `observable_concept_graph.rq` — what makes one Biomedical Concept observable, without the SDTM implementation (example: C105585 Glucose Measurement): the BC side of COSMoS with NCIt, plus the SDTM CT terms that Dataset Specialization rows bind to the BC's DECs (only those bridging triples of the rows).

The boundary of query 2 follows COSMoS's own split into a BC model and an SDTM model. Change the parameter in the VALUES line of a query for another test or BC.

The queries run in Oxigraph (pyoxigraph): rdflib's SPARQL engine takes too long on the "any number of links" path over the 1.5 million triples. The results go to `examples/output/` (not in git: query 1 holds sign-in content when the local `-signin` files are present).

In [ ]:
from collections import Counter
from pathlib import Path

import pandas as pd
from pyoxigraph import DefaultGraph, Quad, RdfFormat, Store

REPO_ROOT = Path.cwd().parent
EXAMPLES = REPO_ROOT / 'examples'
OUT = EXAMPLES / 'output'
RDF_TYPE = 'http://www.w3.org/1999/02/22-rdf-syntax-ns#type'

graph_files = sorted((REPO_ROOT / 'base').glob('*/*.ttl')) + sorted((REPO_ROOT / 'linked').glob('*.ttl'))
store = Store()
for f in graph_files:
    store.bulk_load(path=str(f), format=RdfFormat.TURTLE)
    print(f.relative_to(REPO_ROOT).as_posix())
print(len(store), 'triples')

In [ ]:
def run(query_file):
    """The CONSTRUCT result as a new store, and its triples by rdf:type of the subject."""
    result = Store()
    result.extend(Quad(t.subject, t.predicate, t.object) for t in store.query((EXAMPLES / query_file).read_text(encoding='utf-8')))
    types = Counter(q.object.value for q in result.quads_for_pattern(None, None, None) if q.predicate.value == RDF_TYPE)
    print(query_file, len(result), 'triples')
    print(pd.Series(types, name='nodes').sort_index().to_string())
    return result

def write(result, name):
    OUT.mkdir(exist_ok=True)
    path = OUT / name
    result.dump(output=str(path), format=RdfFormat.N_TRIPLES, from_graph=DefaultGraph())
    print('written', path.relative_to(REPO_ROOT).as_posix())

def table(result, select):
    rows = [[t.value if t is not None else '' for t in s] for s in result.query(select)]
    return pd.DataFrame(rows, columns=[v.value for v in result.query(select).variables])

## 1. Everything linked to the test code GLUC

In [ ]:
findings = run('findings_test_graph.rq')
write(findings, 'findings_test_graph_GLUC.nt')
table(findings, """
SELECT ?dataset_specialization ?bc_id WHERE {
    ?g a <https://www.cdisc.org/cosmos/sdtm_v1.0/SDTMGroup> ;
       <https://www.cdisc.org/cosmos/sdtm_v1.0/datasetSpecializationId> ?dataset_specialization ;
       <https://www.cdisc.org/cosmos/sdtm_v1.0/biomedicalConceptId> ?bc_id .
} ORDER BY ?dataset_specialization""")

## 2. What makes Glucose Measurement observable

In [ ]:
observable = run('observable_concept_graph.rq')
write(observable, 'observable_concept_graph_C105585.nt')
table(observable, """
PREFIX bc: <https://kerfors.github.io/cdisc-for-all/def/cosmos/cdisc_biomedical_concepts_latest#>
PREFIX ct: <https://kerfors.github.io/cdisc-for-all/def/sdtm-ct/SDTM%20Terminology#>
SELECT ?dec_id ?dec_label ?data_type ?example_set ?codelist
       (GROUP_CONCAT(DISTINCT ?term; separator=";") AS ?bound_terms) WHERE {
    ?dec bc:dec_id ?dec_id ; bc:dec_label ?dec_label .
    OPTIONAL { ?dec bc:data_type ?data_type }
    OPTIONAL { ?dec bc:example_set ?example_set }
    OPTIONAL {
        ?row <https://kerfors.github.io/cdisc-for-all/link/cosmos/cdisc_sdtm_dataset_specializations_latest#dec_id> ?dec ;
             <https://kerfors.github.io/cdisc-for-all/link/cosmos/cdisc_sdtm_dataset_specializations_latest#assigned_term> |
             <https://kerfors.github.io/cdisc-for-all/rule/cosmos/cdisc_sdtm_dataset_specializations_latest#value_list/sdtm-ct/SDTM%20Terminology> ?t .
        ?t ct:CDISC%20Submission%20Value ?term ; ct:Codelist%20Name ?codelist .
    }
} GROUP BY ?dec_id ?dec_label ?data_type ?example_set ?codelist ORDER BY ?dec_label ?codelist""")